# AI-Driven Climate & Environmental Monitoring: Africa and USA

This notebook explores five World Bank climate and energy indicators for **29 African countries and the United States**
(30 countries in total) and evaluates a scikit-learn `LinearRegression` forecast of renewable energy share for 2025-2027.

| Indicator | World Bank code | Raw file |
|---|---|---|
| Forest area (% of land area) | `AG.LND.FRST.ZS` | `forest_area.csv` |
| Renewable energy consumption (% of final energy) | `EG.FEC.RNEW.ZS` | `renewable_energy.csv` |
| Agricultural land (% of land area) | `AG.LND.AGRI.ZS` | `agricultural_land.csv` |
| Electric power consumption (kWh per capita) | `EG.USE.ELEC.KH.PC` | `electric_power_consumption.csv` |
| Access to electricity (% of population) | `EG.ELC.ACCS.ZS` | `electricity_access.csv` |

**Prerequisite:** run the pipeline first so the processed files exist:

```bash
python src/transformation/transform.py && python src/loading/load.py
```

In [ ]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Resolve the project root whether the notebook runs from notebooks/ or the repo root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DB_PATH = PROJECT_ROOT / "data" / "climate_monitoring.db"

RAW_FILES = {
    "forest_area": "forest_area.csv",
    "renewable_energy": "renewable_energy.csv",
    "agricultural_land": "agricultural_land.csv",
    "electric_power_consumption": "electric_power_consumption.csv",
    "electricity_access": "electricity_access.csv",
}
print("Project root:", PROJECT_ROOT)

## 1. Load and explore the five raw datasets

Each raw CSV follows the same tidy schema: `country_code, country_name, indicator_id, indicator_name, year, value`.
We check row counts, year coverage, the number of countries and missing values per dataset.

In [ ]:
raw = {name: pd.read_csv(RAW_DIR / fname) for name, fname in RAW_FILES.items()}

overview = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "countries": df["country_code"].nunique(),
        "year_min": int(df["year"].min()),
        "year_max": int(df["year"].max()),
        "null_values": int(df["value"].isna().sum()),
        "value_min": round(df["value"].min(), 2),
        "value_max": round(df["value"].max(), 2),
    }
    for name, df in raw.items()
])
overview

In [ ]:
raw["renewable_energy"].head(10)

In [ ]:
# Countries missing from the electric power consumption series (World Bank gaps)
all_codes = set(raw["forest_area"]["country_code"])
missing_power = sorted(all_codes - set(raw["electric_power_consumption"]["country_code"]))
print("Countries without electric power consumption data:", missing_power)

## 2. Processed, analysis-ready data

`src/transformation/transform.py` normalises country names (e.g. *Egypt, Arab Rep.* -> *Egypt*), casts types,
deduplicates, pivots to one row per country-year and adds derived features:

- `renewable_energy_trend` - year-over-year change in renewable share (percentage points)
- `forest_loss_rate` - annual % loss of forest area (positive = forest lost)
- `electricity_access_change`, `agricultural_land_change` - YoY changes in percentage points

In [ ]:
wide = pd.read_csv(PROCESSED_DIR / "climate_indicators_wide.csv")
profiles = pd.read_csv(PROCESSED_DIR / "country_profiles.csv")
predictions = pd.read_csv(PROCESSED_DIR / "ml_predictions.csv")

print("wide:", wide.shape, "| profiles:", profiles.shape, "| predictions:", predictions.shape)
wide.describe().T.round(2)

In [ ]:
# Confirm the SQLite warehouse matches the processed files
with sqlite3.connect(DB_PATH) as conn:
    counts = {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
              for t in ["climate_indicators", "country_profiles", "ml_predictions"]}
counts

## 3. Renewable energy trends: top 10 African countries vs USA (2012-2022)

We select the ten African countries with the highest latest renewable share and plot them alongside the USA.
Note that in much of Sub-Saharan Africa the "renewable" share is dominated by **traditional biomass** (firewood,
charcoal), not modern solar or wind - so a high share is not by itself a sign of a clean, modern energy system.

In [ ]:
renew = wide.dropna(subset=["renewable_energy_pct"])
renew = renew[(renew["year"] >= 2012) & (renew["year"] <= 2022)]

top10_africa = (profiles[profiles["region"] == "Africa"]
                .nlargest(10, "latest_renewable_energy_pct")["country_name"].tolist())
selection = top10_africa + ["United States"]

fig = px.line(
    renew[renew["country_name"].isin(selection)],
    x="year", y="renewable_energy_pct", color="country_name", markers=True,
    title="Renewable energy share of final energy consumption, 2012-2022",
    labels={"renewable_energy_pct": "Renewable energy (%)", "year": "Year", "country_name": "Country"},
)
fig.update_layout(height=550, legend_title_text="Country")
fig.show()

## 4. Forest cover comparison (latest year)

Forest area as a share of land area, latest available year, with the USA highlighted.

In [ ]:
forest_latest = profiles.sort_values("latest_forest_area_pct", ascending=False)
fig = px.bar(
    forest_latest, x="country_name", y="latest_forest_area_pct", color="region",
    color_discrete_map={"Africa": "#2E8B57", "USA": "#1F4E79"},
    hover_data=["latest_forest_area_pct_year", "change_forest_area_pct", "avg_forest_loss_rate"],
    title="Forest area (% of land area), latest available year",
    labels={"latest_forest_area_pct": "Forest area (%)", "country_name": "Country", "region": "Region"},
)
fig.update_layout(height=550, xaxis_tickangle=-45)
fig.show()

In [ ]:
forest_trend = (wide.dropna(subset=["forest_area_pct"])
                .groupby(["year", "region"])["forest_area_pct"].mean()
                .unstack().round(2))
forest_trend["africa_minus_usa"] = (forest_trend["Africa"] - forest_trend["USA"]).round(2)
forest_trend

## 5. Electricity access vs renewable energy

Each point is a country-year. The strong negative relationship is the **energy poverty paradox**: countries with the
lowest grid access report the highest "renewable" share because households rely on biomass. The USA sits at the
opposite corner (universal access, ~10-11% renewables).

In [ ]:
scatter_df = wide.dropna(subset=["renewable_energy_pct", "electricity_access_pct"])
fig = px.scatter(
    scatter_df, x="electricity_access_pct", y="renewable_energy_pct",
    color="sub_region", hover_name="country_name", hover_data=["year"],
    trendline=None,
    title="Access to electricity vs renewable energy share (country-years)",
    labels={"electricity_access_pct": "Access to electricity (% of population)",
            "renewable_energy_pct": "Renewable energy (%)", "sub_region": "Sub-region"},
)
# Ordinary least squares line computed with numpy for transparency
slope, intercept = np.polyfit(scatter_df["electricity_access_pct"], scatter_df["renewable_energy_pct"], 1)
xs = np.linspace(scatter_df["electricity_access_pct"].min(), scatter_df["electricity_access_pct"].max(), 50)
fig.add_trace(go.Scatter(x=xs, y=slope * xs + intercept, mode="lines",
                         name=f"OLS fit (slope={slope:.2f})", line=dict(color="black", dash="dash")))
fig.update_layout(height=550)
fig.show()

r = scatter_df["electricity_access_pct"].corr(scatter_df["renewable_energy_pct"])
print(f"Pearson r (access vs renewables): {r:.3f} over {len(scatter_df)} country-years")

## 6. Correlation matrix of climate indicators

Pearson correlations across all country-years for the five indicators and the engineered trend features.

In [ ]:
corr_cols = ["forest_area_pct", "renewable_energy_pct", "agricultural_land_pct",
             "electric_power_kwh_pc", "electricity_access_pct",
             "renewable_energy_trend", "forest_loss_rate"]
corr = wide[corr_cols].corr().round(2)

fig = px.imshow(
    corr, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="Correlation matrix of climate & energy indicators",
)
fig.update_layout(height=600, width=750)
fig.show()

## 7. Agricultural land pressure by country

Agricultural land share (latest year) on a choropleth, plus a ranked bar chart. Agricultural expansion is a key
driver of forest loss in West and East Africa.

In [ ]:
fig = px.choropleth(
    profiles, locations="country_code", color="latest_agricultural_land_pct",
    hover_name="country_name", color_continuous_scale="YlOrBr",
    title="Agricultural land (% of land area), latest year",
    labels={"latest_agricultural_land_pct": "Agricultural land (%)"},
)
fig.update_geos(showcountries=True, projection_type="natural earth")
fig.update_layout(height=550)
fig.show()

agri_sorted = profiles.sort_values("latest_agricultural_land_pct", ascending=True)
fig = px.bar(
    agri_sorted, x="latest_agricultural_land_pct", y="country_name", orientation="h",
    color="change_agricultural_land_pct", color_continuous_scale="RdYlGn_r",
    title="Agricultural land (% of land area) and change since first observation",
    labels={"latest_agricultural_land_pct": "Agricultural land (%)", "country_name": "",
            "change_agricultural_land_pct": "Change (pp)"},
)
fig.update_layout(height=800)
fig.show()

## 8. ML forecast: renewable energy share 2025-2027

A per-country `LinearRegression` (`renewable_energy_pct ~ year`) is trained on all available history (2013-2022).
The chart shows actuals (solid) and forecasts (dashed) for selected countries. In-sample R² values are reported
so that low-confidence (flat or noisy) trends can be identified.

In [ ]:
focus = ["United States", "South Africa", "Kenya", "Ghana", "Rwanda", "Angola", "Egypt", "Nigeria"]
fig = go.Figure()
palette = px.colors.qualitative.Plotly
for i, country in enumerate(focus):
    color = palette[i % len(palette)]
    hist = wide[(wide["country_name"] == country)].dropna(subset=["renewable_energy_pct"])
    pred = predictions[predictions["country_name"] == country].sort_values("forecast_year")
    if hist.empty or pred.empty:
        continue
    fig.add_trace(go.Scatter(x=hist["year"], y=hist["renewable_energy_pct"], mode="lines+markers",
                             name=f"{country} (actual)", line=dict(color=color)))
    bridge_x = [hist["year"].iloc[-1]] + pred["forecast_year"].tolist()
    bridge_y = [hist["renewable_energy_pct"].iloc[-1]] + pred["predicted_renewable_energy_pct"].tolist()
    fig.add_trace(go.Scatter(x=bridge_x, y=bridge_y, mode="lines+markers",
                             name=f"{country} (forecast)", line=dict(color=color, dash="dash")))
fig.update_layout(title="Renewable energy share: actuals and LinearRegression forecasts to 2027",
                  xaxis_title="Year", yaxis_title="Renewable energy (%)", height=600)
fig.show()

In [ ]:
forecast_table = (predictions
                  .pivot_table(index=["country_name", "region", "r2_score", "trend_slope_pp_per_year"],
                               columns="forecast_year", values="predicted_renewable_energy_pct")
                  .reset_index()
                  .sort_values(2027, ascending=False))
forecast_table.columns = [str(c) for c in forecast_table.columns]
print("Model quality - median in-sample R²:", round(predictions["r2_score"].median(), 3))
forecast_table.round(2)

## 9. Africa vs USA: headline comparison

In [ ]:
latest_cols = ["latest_forest_area_pct", "latest_renewable_energy_pct", "latest_agricultural_land_pct",
               "latest_electric_power_kwh_pc", "latest_electricity_access_pct", "avg_forest_loss_rate"]
comparison = profiles.groupby("region")[latest_cols].mean().T.round(2)
comparison.columns.name = None
comparison

## 10. Key insights - Africa vs USA

1. **Forest loss is concentrated in Africa.** Average forest cover across the African sample fell from about 29.1% (2015)
   to 27.7% (2024), an average loss rate of roughly 0.5% of remaining forest per year, while US forest cover was nearly
   flat (33.9% -> 33.8%, ~0.02-0.05% per year).
2. **The energy poverty paradox.** Renewable share and electricity access are strongly negatively correlated
   (Pearson r ≈ -0.82 across country-years). Liberia, Uganda and Ethiopia exceed 90% renewables largely through biomass,
   while the USA, with universal access, sits near 11%.
3. **Rapid electrification in East Africa.** Rwanda (+49 pp), Uganda (+37 pp) and Kenya (+35 pp) recorded the largest
   gains in electricity access between 2015 and 2024; Niger (21%) and Burkina Faso (34%) remain lowest.
4. **Diverging renewable trajectories.** The linear models project the USA (≈12.6% by 2027), South Africa and Egypt
   continuing slow increases in renewables from a fossil-heavy base, while many biomass-dependent West African
   countries (Côte d'Ivoire, Ghana, Senegal, Mali) trend downward as modern - often fossil - energy displaces firewood.
5. **Agricultural pressure.** Agricultural land exceeds 70% of land area in several West/East African countries
   (Nigeria, Côte d'Ivoire, Rwanda, Gambia), linking land-use change directly to forest decline.
6. **Model caveat.** Linear trends on ~9-10 annual points are a transparent baseline, not a definitive forecast; countries
   with low R² (e.g. Nigeria, Zambia, Guinea-Bissau) have essentially flat or noisy histories.